In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
from torchvision import transforms

image_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),   # Unified input size
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],   # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ]),
    
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}


In [3]:
data_dir = "/kaggle/input/xray-ds/split_dataset"

train_dataset = datasets.ImageFolder(
    root=f"{data_dir}/train",
    transform=image_transforms['train']
)
val_dataset = datasets.ImageFolder(
    root=f"{data_dir}/val",
    transform=image_transforms['val']
)
test_dataset = datasets.ImageFolder(
    root=f"{data_dir}/test",
    transform=image_transforms['test']
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)


In [4]:
class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)
print("Num classes:", num_classes)


['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Num classes: 4


In [5]:
# ==========================================
# CLASS WEIGHTS COMPUTATION (TRAIN SET ONLY)
# ==========================================

def get_class_counts(dataset):
    """
    Computes number of samples per class from ImageFolder dataset.
    """
    targets = dataset.targets
    class_to_idx = dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    counts = {}
    for idx in idx_to_class:
        counts[idx] = targets.count(idx)

    return counts


# Compute counts from TRAIN dataset only
class_counts = get_class_counts(train_dataset)

classes = list(class_counts.keys())
counts = list(class_counts.values())

total_samples = sum(counts)
num_classes = len(classes)

weights = []
for i in range(num_classes):
    weight = total_samples / (num_classes * counts[i])
    weights.append(weight)

class_weights_tensor = torch.FloatTensor(weights).to(device)

print("Class counts:", class_counts)
print("Class weights:", weights)


Class counts: {0: 2892, 1: 4809, 2: 8153, 3: 1076}
Class weights: [1.4635200553250345, 0.880120607194843, 0.5191340610818104, 3.933550185873606]


In [6]:
# ==========================================
# TINY CNN MICRO-ARCHITECTURE
# ==========================================

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.act(x)
        return x


class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(16, 32),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(32, 64),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(64, 96),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(96, 128),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = TinyCNN(num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [7]:
sum(p.numel() for p in model.parameters())

32484

In [8]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

outputs = model(images)
print(outputs.shape, labels.min().item(), labels.max().item())


torch.Size([32, 4]) 0 3


In [9]:
def train_model_with_early_stopping(
    model,
    criterion,
    optimizer,
    num_epochs=20,
    patience=5
):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # ================= TRAIN =================
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels)

        epoch_train_loss = running_loss / len(train_dataset)
        epoch_train_acc = running_corrects.double() / len(train_dataset)

        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc.cpu())

        # ================= VALIDATION =================
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

        epoch_val_loss = running_loss / len(val_dataset)
        epoch_val_acc = running_corrects.double() / len(val_dataset)

        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc.cpu())

        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f}")
        print(f"Val   Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        # ================= EARLY STOPPING =================
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_densenet121.pth")
        else:
            patience_counter += 1
            print(f"EarlyStopping counter: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    return model, train_losses, val_losses, train_accs, val_accs


In [10]:
model, train_losses, val_losses, train_accs, val_accs = \
    train_model_with_early_stopping(
        model,
        criterion,
        optimizer,
        num_epochs=20,
        patience=5
    )



Epoch 1/20
------------------------------
Train Loss: 1.1893 Acc: 0.5673
Val   Loss: 1.0080 Acc: 0.6359

Epoch 2/20
------------------------------
Train Loss: 0.8258 Acc: 0.6395
Val   Loss: 0.8429 Acc: 0.6794

Epoch 3/20
------------------------------
Train Loss: 0.6826 Acc: 0.6921
Val   Loss: 0.7448 Acc: 0.7050

Epoch 4/20
------------------------------
Train Loss: 0.5916 Acc: 0.7307
Val   Loss: 0.5717 Acc: 0.8019

Epoch 5/20
------------------------------
Train Loss: 0.5141 Acc: 0.7660
Val   Loss: 0.6639 Acc: 0.7234
EarlyStopping counter: 1/5

Epoch 6/20
------------------------------
Train Loss: 0.4700 Acc: 0.7897
Val   Loss: 0.4907 Acc: 0.8184

Epoch 7/20
------------------------------
Train Loss: 0.4293 Acc: 0.8035
Val   Loss: 0.4435 Acc: 0.8336

Epoch 8/20
------------------------------
Train Loss: 0.4020 Acc: 0.8152
Val   Loss: 0.3899 Acc: 0.8525

Epoch 9/20
------------------------------
Train Loss: 0.3820 Acc: 0.8259
Val   Loss: 0.4375 Acc: 0.8416
EarlyStopping counter: 1/5



In [11]:
def evaluate_test_accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_acc = correct / total
    return test_acc


In [12]:
test_accuracy = evaluate_test_accuracy(model, test_loader, device)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


Test Accuracy: 82.50%


In [13]:
from collections import defaultdict

def per_class_accuracy(model, dataloader, class_names, device):
    model.eval()
    correct = defaultdict(int)
    total = defaultdict(int)

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            for label, pred in zip(labels, preds):
                total[label.item()] += 1
                if label == pred:
                    correct[label.item()] += 1

    for idx, name in enumerate(class_names):
        acc = correct[idx] / total[idx]
        print(f"{name}: {acc * 100:.2f}%")


In [14]:
per_class_accuracy(model, test_loader, class_names, device)


COVID: 98.90%
Lung_Opacity: 83.39%
Normal: 73.92%
Viral Pneumonia: 99.26%


In [15]:
import os

PLOT_DIR = "/kaggle/working/plots"
os.makedirs(PLOT_DIR, exist_ok=True)
import matplotlib.pyplot as plt

def save_loss_plot(train_losses, val_losses):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/loss_curve.png", dpi=300, bbox_inches="tight")
    plt.close()


In [16]:
def save_accuracy_plot(train_accs, val_accs):
    plt.figure()
    plt.plot(train_accs, label="Train Accuracy")
    plt.plot(val_accs, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/accuracy_curve.png", dpi=300, bbox_inches="tight")
    plt.close()


In [17]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

def save_confusion_matrix(model, dataloader, class_names, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d",
        xticklabels=class_names,
        yticklabels=class_names,
        cmap="Blues"
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.savefig(f"{PLOT_DIR}/confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()


In [18]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def save_roc_curves(model, dataloader, class_names, device):
    model.eval()
    y_true, y_scores = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)

            y_true.extend(labels.numpy())
            y_scores.extend(torch.softmax(outputs, dim=1).cpu().numpy())

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

    plt.figure(figsize=(8, 6))
    for i, class_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{class_name} (AUC={roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Multi-class ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/roc_curve.png", dpi=300, bbox_inches="tight")
    plt.close()


In [19]:
save_loss_plot(train_losses, val_losses)
save_accuracy_plot(train_accs, val_accs)

save_confusion_matrix(
    model,
    test_loader,
    class_names,
    device
)

save_roc_curves(
    model,
    test_loader,
    class_names,
    device
)
